In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    concat_ws,
    current_timestamp,
    desc,
    from_utc_timestamp,
    lit,
    row_number,
    sha2,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
from pyspark.sql.window import Window

In [0]:
def load_curated_firm_structure_fldt(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #    "FirmStructureId string,"
        #    "SourceSystem string,"
        #    "IngestionTimestamp timestamp,"
        #    "BatchId string,"
        #    "FirmStructureKey string,"
        #    "EmployeePartyKey string,"
        #    "EmployeeCategory string,"
        #    "EmployeeFirstName string,"
        #     "EmployeeLastName string,"
        #     "EmployeeFullName string,"
        #    "EmployeeEmail string,"
        #    "EmployeeUsername string,"
        #     "EmployeeUserTypeCode string,"
        #    "EmployeeUserType string,"
        #    "EmployeePositionCode string ,"
        #    "EmployeePosition string,"
        #    "DepartmentCode string,"
        #    "Department string,"
        #    "TeamCode string,"
        #    "Team string,"
        #    "SupervisorPartyKey string,"
        #     "SupervisorPartyName string,"
        #     "SupervisorUsername string,"
        #    "ManagerPartyKey string,"
        #    "ManagerPartyName string,"
        #    "ManagerUsername string,"
        #    "DirectorPartyKey string,"
        #    "DirectorPartyName string,"
        #    "DirectorUsername string,"
        #     "IsEmployeeActiveFlag boolean,"
        #    "CreatedDate date,"
        #    "LastUpdatedDate date,"
        #    "BusinessDate date"
        # )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.FirmStructure"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")


        logging.info("Delta Table has been created!")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "fldt",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # =================== Application ===========================

        application_extra_filters = None

        if application_extra_filters is not None:
            application_extra_filters.append(date_filter)
            application_final_filters = " AND ".join(application_extra_filters)
        else:
            application_final_filters = date_filter

        application_selected_fields = [
            "AssigneeUserName",
            "AssigneeUserId",
            "AssigneeUserFullName",
            "UserCreated",
            "UserCreatedFullName",
            "ProcessingUserName",
            "ProcessingUserId",
            "ProcessingUserFullName",
            "DateUpdated",
            "BusinessDate",
        ]

        application = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "fldt_application",
            application_final_filters,
            application_selected_fields,
        )

        # ================= CompletedApplicationStep =============

        completed_application_step_extra_filters = None

        if completed_application_step_extra_filters is not None:
            completed_application_step_extra_filters.append(date_filter)
            completed_application_step_final_filters = " AND ".join(
                completed_application_step_extra_filters
            )
        else:
            completed_application_step_final_filters = date_filter

        completed_application_step_selected_fields = [
            "AssigneeUserName",
            "AssigneeUserId",
            "AssigneeUserFullName",
            "ExecutedByUserName",
            "ExecutedByUserId",
            "ExecutedByUserFullName",
            "ExecutionDate",
            "BusinessDate",
        ]

        completed_application_step = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "fldt_CompletedApplicationStep",
            completed_application_step_final_filters,
            completed_application_step_selected_fields,
        )

        # ============== FinalSolution ===================

        final_solution_extra_filters = None

        if final_solution_extra_filters is not None:
            final_solution_extra_filters.append(date_filter)
            final_solution_final_filters = " AND ".join(final_solution_extra_filters)
        else:
            final_solution_final_filters = date_filter

        final_solution_selected_fields = [
            "UserName",
            "FullName",
            "DateInserted",
            "BusinessDate",
        ]

        final_solution = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "fldt_finalsolution",
            final_solution_final_filters,
            final_solution_selected_fields,
        )

        # ============ firmstructure_first_partition_df ===============
        application_first_partition = application.select(
            "AssigneeUserName",
            "AssigneeUserId",
            "AssigneeUserFullName",
            "DateUpdated",
            "BusinessDate",
        ).distinct()

        window_spec_first = Window.partitionBy(
            application_first_partition.AssigneeUserName
        ).orderBy(desc(application_first_partition.DateUpdated))

        firm_structure_fldt_first_partition_primary_key_fields = [
            application_first_partition.AssigneeUserName,
            lit("fldt"),
            lit("Employee"),
        ]

        firm_structure_fldt_first_partition_firmstructurekey_fields = [
            application_first_partition.AssigneeUserName,
            lit("Employee"),
        ]

        application_first_partition = application_first_partition.withColumn(
            "rn", row_number().over(window_spec_first)
        )

        application_first_partition = application_first_partition.filter(col("rn") == 1)

        firm_structure_first_partition = application_first_partition.withColumn(
            "FirmStructureId",
            sha2(
                concat_ws(";", *firm_structure_fldt_first_partition_primary_key_fields),
                256,
            ),
        ).withColumn(
            "FirmStructureKey",
            concat_ws(
                ";", *firm_structure_fldt_first_partition_firmstructurekey_fields
            ),
        )

        firm_structure_first_partition_df = firm_structure_first_partition.select(
            firm_structure_first_partition.FirmStructureId.cast("string").alias(
                "FirmStructureId"
            ),
            lit("fldt").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            firm_structure_first_partition.FirmStructureKey.cast("string").alias(
                "FirmStructureKey"
            ),
            application_first_partition.AssigneeUserId.cast("string").alias(
                "EmployeePartyKey"
            ),
            lit("Employee").alias("EmployeeCategory"),
            lit(None).alias("EmployeeFirstName"),
            lit(None).alias("EmployeeLastName"),
            application_first_partition.AssigneeUserFullName.cast("string").alias(
                "EmployeeFullName"
            ),
            lit(None).alias("EmployeeEmail"),
            application_first_partition.AssigneeUserName.cast("string").alias(
                "EmployeeUserName"
            ),
            lit(None).alias("EmployeeUserTypeCode"),
            lit(None).alias("EmployeeUserType"),
            lit(None).alias("EmployeePositionCode"),
            lit(None).alias("EmployeePosition"),
            lit(None).alias("DepartmentCode"),
            lit(None).alias("Department"),
            lit(None).alias("TeamCode"),
            lit(None).alias("Team"),
            lit(None).alias("SupervisorPartyKey"),
            lit(None).alias("SupervisorPartyName"),
            lit(None).alias("SupervisorUserName"),
            lit(None).alias("ManagerPartyKey"),
            lit(None).alias("ManagerPartyName"),
            lit(None).alias("DirectorUsername"),
            lit(None).alias("IsEmployeeActiveFlag"),
            lit(None).alias("CreatedDate"),
            lit(None).alias("LastUpdatedDate"),
            application_first_partition.BusinessDate.alias("BusinessDate"),
        )

        # ================ firm_structure_second_partition_df ================

        completed_application_step_second_partition = completed_application_step.select(
            "AssigneeUserName",
            "AssigneeUserId",
            "AssigneeUserFullName",
            "ExecutionDate",
            "BusinessDate",
        ).distinct()

        window_spec_second = Window.partitionBy(
            completed_application_step_second_partition.AssigneeUserName
        ).orderBy(desc(completed_application_step_second_partition.ExecutionDate))

        firm_structure_fldt_second_partition_primary_key_fields = [
            completed_application_step_second_partition.AssigneeUserName,
            lit("fldt"),
            lit("Employee"),
        ]

        firm_structure_fldt_second_partition_firmstructurekey_fields = [
            completed_application_step_second_partition.AssigneeUserName,
            lit("Employee"),
        ]

        completed_application_step_second_partition = (
            completed_application_step_second_partition.withColumn(
                "rn", row_number().over(window_spec_second)
            )
        )

        completed_application_step_second_partition = (
            completed_application_step_second_partition.filter(col("rn") == 1)
        )

        firm_structure_second_partition = (
            completed_application_step_second_partition.withColumn(
                "FirmStructureId",
                sha2(
                    concat_ws(
                        ";", *firm_structure_fldt_second_partition_primary_key_fields
                    ),
                    256,
                ),
            ).withColumn(
                "FirmStructureKey",
                concat_ws(
                    ";", *firm_structure_fldt_second_partition_firmstructurekey_fields
                ),
            )
        )

        firm_structure_second_partition_df = firm_structure_second_partition.select(
            firm_structure_second_partition.FirmStructureId.cast("string").alias(
                "FirmStructureId"
            ),
            lit("fldt").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            firm_structure_second_partition.FirmStructureKey.cast("string").alias(
                "FirmStructureKey"
            ),
            completed_application_step_second_partition.AssigneeUserId.cast(
                "string"
            ).alias("EmployeePartyKey"),
            lit("Employee").alias("EmployeeCategory"),
            lit(None).alias("EmployeeFirstName"),
            lit(None).alias("EmployeeLastName"),
            completed_application_step_second_partition.AssigneeUserFullName.cast(
                "string"
            ).alias("EmployeeFullName"),
            lit(None).alias("EmployeeEmail"),
            completed_application_step_second_partition.AssigneeUserName.cast(
                "string"
            ).alias("EmployeeUserName"),
            lit(None).alias("EmployeeUserTypeCode"),
            lit(None).alias("EmployeeUserType"),
            lit(None).alias("EmployeePositionCode"),
            lit(None).alias("EmployeePosition"),
            lit(None).alias("DepartmentCode"),
            lit(None).alias("Department"),
            lit(None).alias("TeamCode"),
            lit(None).alias("Team"),
            lit(None).alias("SupervisorPartyKey"),
            lit(None).alias("SupervisorPartyName"),
            lit(None).alias("SupervisorUserName"),
            lit(None).alias("ManagerPartyKey"),
            lit(None).alias("ManagerPartyName"),
            lit(None).alias("DirectorUsername"),
            lit(None).alias("IsEmployeeActiveFlag"),
            lit(None).alias("CreatedDate"),
            lit(None).alias("LastUpdatedDate"),
            completed_application_step_second_partition.BusinessDate.alias(
                "BusinessDate"
            ),
        )

        # ================= firm_structure_third_partition_df ==============

        completed_application_step_third_partition = completed_application_step.select(
            "ExecutedByUserName",
            "ExecutedByUserId",
            "ExecutedByUserFullName",
            "ExecutionDate",
            "BusinessDate",
        ).distinct()

        window_spec_third = Window.partitionBy(
            completed_application_step_third_partition.ExecutedByUserName
        ).orderBy(desc(completed_application_step_third_partition.ExecutionDate))

        firm_structure_fldt_third_partition_primary_key_fields = [
            completed_application_step_third_partition.ExecutedByUserName,
            lit("fldt"),
            lit("Employee"),
        ]

        firm_structure_fldt_third_partition_firmstructurekey_fields = [
            completed_application_step_third_partition.ExecutedByUserName,
            lit("Employee"),
        ]

        completed_application_step_third_partition = (
            completed_application_step_third_partition.withColumn(
                "rn", row_number().over(window_spec_third)
            )
        )

        completed_application_step_third_partition = (
            completed_application_step_third_partition.filter(col("rn") == 1)
        )

        firm_structure_third_partition = (
            completed_application_step_third_partition.withColumn(
                "FirmStructureId",
                sha2(
                    concat_ws(
                        ";", *firm_structure_fldt_third_partition_primary_key_fields
                    ),
                    256,
                ),
            ).withColumn(
                "FirmStructureKey",
                concat_ws(
                    ";", *firm_structure_fldt_third_partition_firmstructurekey_fields
                ),
            )
        )

        firm_structure_third_partition_df = firm_structure_third_partition.select(
            firm_structure_third_partition.FirmStructureId.cast("string").alias(
                "FirmStructureId"
            ),
            lit("fldt").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            firm_structure_third_partition.FirmStructureKey.cast("string").alias(
                "FirmStructureKey"
            ),
            completed_application_step_third_partition.ExecutedByUserId.cast(
                "string"
            ).alias("EmployeePartyKey"),
            lit("Employee").alias("EmployeeCategory"),
            lit(None).alias("EmployeeFirstName"),
            lit(None).alias("EmployeeLastName"),
            completed_application_step_third_partition.ExecutedByUserFullName.cast(
                "string"
            ).alias("EmployeeFullName"),
            lit(None).alias("EmployeeEmail"),
            completed_application_step_third_partition.ExecutedByUserName.cast(
                "string"
            ).alias("EmployeeUserName"),
            lit(None).alias("EmployeeUserTypeCode"),
            lit(None).alias("EmployeeUserType"),
            lit(None).alias("EmployeePositionCode"),
            lit(None).alias("EmployeePosition"),
            lit(None).alias("DepartmentCode"),
            lit(None).alias("Department"),
            lit(None).alias("TeamCode"),
            lit(None).alias("Team"),
            lit(None).alias("SupervisorPartyKey"),
            lit(None).alias("SupervisorPartyName"),
            lit(None).alias("SupervisorUserName"),
            lit(None).alias("ManagerPartyKey"),
            lit(None).alias("ManagerPartyName"),
            lit(None).alias("DirectorUsername"),
            lit(None).alias("IsEmployeeActiveFlag"),
            lit(None).alias("CreatedDate"),
            lit(None).alias("LastUpdatedDate"),
            completed_application_step_third_partition.BusinessDate.alias(
                "BusinessDate"
            ),
        )

        # ================== firm_structure_fourth_partition_df ===============

        application_fourth_partition = application.select(
            "ProcessingUserName",
            "ProcessingUserId",
            "ProcessingUserFullName",
            "DateUpdated",
            "BusinessDate",
        ).distinct()

        window_spec_fourth = Window.partitionBy(
            application_fourth_partition.ProcessingUserName
        ).orderBy(desc(application_fourth_partition.DateUpdated))

        firm_structure_fldt_fourth_partition_primary_key_fields = [
            application_fourth_partition.ProcessingUserName,
            lit("fldt"),
            lit("Employee"),
        ]

        firm_structure_fldt_fourth_partition_firmstructurekey_fields = [
            application_fourth_partition.ProcessingUserName,
            lit("Employee"),
        ]

        application_fourth_partition = application_fourth_partition.withColumn(
            "rn", row_number().over(window_spec_fourth)
        )

        application_fourth_partition = application_fourth_partition.filter(
            col("rn") == 1
        )

        firm_structure_fourth_partition = application_fourth_partition.withColumn(
            "FirmStructureId",
            sha2(
                concat_ws(
                    ";", *firm_structure_fldt_fourth_partition_primary_key_fields
                ),
                256,
            ),
        ).withColumn(
            "FirmStructureKey",
            concat_ws(
                ";", *firm_structure_fldt_fourth_partition_firmstructurekey_fields
            ),
        )

        firm_structure_fourth_partition_df = firm_structure_fourth_partition.select(
            firm_structure_fourth_partition.FirmStructureId.cast("string").alias(
                "FirmStructureId"
            ),
            lit("fldt").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            firm_structure_fourth_partition.FirmStructureKey.cast("string").alias(
                "FirmStructureKey"
            ),
            application_fourth_partition.ProcessingUserId.cast("string").alias(
                "EmployeePartyKey"
            ),
            lit("Employee").alias("EmployeeCategory"),
            lit(None).alias("EmployeeFirstName"),
            lit(None).alias("EmployeeLastName"),
            application_fourth_partition.ProcessingUserFullName.cast("string").alias(
                "EmployeeFullName"
            ),
            lit(None).alias("EmployeeEmail"),
            application_fourth_partition.ProcessingUserName.cast("string").alias(
                "EmployeeUserName"
            ),
            lit(None).alias("EmployeeUserTypeCode"),
            lit(None).alias("EmployeeUserType"),
            lit(None).alias("EmployeePositionCode"),
            lit(None).alias("EmployeePosition"),
            lit(None).alias("DepartmentCode"),
            lit(None).alias("Department"),
            lit(None).alias("TeamCode"),
            lit(None).alias("Team"),
            lit(None).alias("SupervisorPartyKey"),
            lit(None).alias("SupervisorPartyName"),
            lit(None).alias("SupervisorUserName"),
            lit(None).alias("ManagerPartyKey"),
            lit(None).alias("ManagerPartyName"),
            lit(None).alias("DirectorUsername"),
            lit(None).alias("IsEmployeeActiveFlag"),
            lit(None).alias("CreatedDate"),
            lit(None).alias("LastUpdatedDate"),
            application_fourth_partition.BusinessDate.alias("BusinessDate"),
        )

        # ================== firm_structure_fifth_partition_df ===============

        application_fifth_partition = application.select(
            "UserCreated", "UserCreatedFullName", "DateUpdated", "BusinessDate"
        ).distinct()

        window_spec_fifth = Window.partitionBy(
            application_fifth_partition.UserCreated
        ).orderBy(desc(application_fifth_partition.DateUpdated))

        firm_structure_fldt_fifth_partition_primary_key_fields = [
            application_fifth_partition.UserCreated,
            lit("fldt"),
            lit("Employee"),
        ]

        firm_structure_fldt_fifth_partition_firmstructurekey_fields = [
            application_fifth_partition.UserCreated,
            lit("Employee"),
        ]

        application_fifth_partition = application_fifth_partition.withColumn(
            "rn", row_number().over(window_spec_fifth)
        )

        application_fifth_partition = application_fifth_partition.filter(col("rn") == 1)

        firm_structure_fifth_partition = application_fifth_partition.withColumn(
            "FirmStructureId",
            sha2(
                concat_ws(";", *firm_structure_fldt_fifth_partition_primary_key_fields),
                256,
            ),
        ).withColumn(
            "FirmStructureKey",
            concat_ws(
                ";", *firm_structure_fldt_fifth_partition_firmstructurekey_fields
            ),
        )

        firm_structure_fifth_partition_df = firm_structure_fifth_partition.select(
            firm_structure_fifth_partition.FirmStructureId.cast("string").alias(
                "FirmStructureId"
            ),
            lit("fldt").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            firm_structure_fifth_partition.FirmStructureKey.cast("string").alias(
                "FirmStructureKey"
            ),
            lit(None).alias("EmployeePartyKey"),
            lit("Employee").alias("EmployeeCategory"),
            lit(None).alias("EmployeeFirstName"),
            lit(None).alias("EmployeeLastName"),
            application_fifth_partition.UserCreatedFullName.cast("string").alias(
                "EmployeeFullName"
            ),
            lit(None).alias("EmployeeEmail"),
            application_fifth_partition.UserCreated.cast("string").alias(
                "EmployeeUserName"
            ),
            lit(None).alias("EmployeeUserTypeCode"),
            lit(None).alias("EmployeeUserType"),
            lit(None).alias("EmployeePositionCode"),
            lit(None).alias("EmployeePosition"),
            lit(None).alias("DepartmentCode"),
            lit(None).alias("Department"),
            lit(None).alias("TeamCode"),
            lit(None).alias("Team"),
            lit(None).alias("SupervisorPartyKey"),
            lit(None).alias("SupervisorPartyName"),
            lit(None).alias("SupervisorUserName"),
            lit(None).alias("ManagerPartyKey"),
            lit(None).alias("ManagerPartyName"),
            lit(None).alias("DirectorUsername"),
            lit(None).alias("IsEmployeeActiveFlag"),
            lit(None).alias("CreatedDate"),
            lit(None).alias("LastUpdatedDate"),
            application_fifth_partition.BusinessDate.alias("BusinessDate"),
        )

        # ================= firm_structure_sixth_partition_df ========

        final_solution_sixth_partition = final_solution.select(
            "UserName", "FullName", "DateInserted", "BusinessDate"
        ).distinct()

        window_spec_sixth = Window.partitionBy(
            final_solution_sixth_partition.UserName
        ).orderBy(desc(final_solution_sixth_partition.DateInserted))

        firm_structure_fldt_sixth_partition_primary_key_fields = [
            final_solution_sixth_partition.UserName,
            lit("fldt"),
            lit("Employee"),
        ]

        firm_structure_fldt_sixth_partition_firmstructurekey_fields = [
            final_solution_sixth_partition.UserName,
            lit("Employee"),
        ]

        final_solution_sixth_partition = final_solution_sixth_partition.withColumn(
            "rn", row_number().over(window_spec_sixth)
        )

        final_solution_sixth_partition = final_solution_sixth_partition.filter(
            col("rn") == 1
        )

        firm_structure_sixth_partition = final_solution_sixth_partition.withColumn(
            "FirmStructureId",
            sha2(
                concat_ws(";", *firm_structure_fldt_sixth_partition_primary_key_fields),
                256,
            ),
        ).withColumn(
            "FirmStructureKey",
            concat_ws(
                ";", *firm_structure_fldt_sixth_partition_firmstructurekey_fields
            ),
        )

        firm_structure_sixth_partition_df = firm_structure_sixth_partition.select(
            firm_structure_sixth_partition.FirmStructureId.cast("string").alias(
                "FirmStructureId"
            ),
            lit("fldt").alias("SourceSystem"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).alias("BatchId"),
            firm_structure_sixth_partition.FirmStructureKey.cast("string").alias(
                "FirmStructureKey"
            ),
            lit(None).alias("EmployeePartyKey"),
            lit("Employee").alias("EmployeeCategory"),
            lit(None).alias("EmployeeFirstName"),
            lit(None).alias("EmployeeLastName"),
            final_solution_sixth_partition.FullName.cast("string").alias(
                "EmployeeFullName"
            ),
            lit(None).alias("EmployeeEmail"),
            final_solution_sixth_partition.UserName.cast("string").alias(
                "EmployeeUserName"
            ),
            lit(None).alias("EmployeeUserTypeCode"),
            lit(None).alias("EmployeeUserType"),
            lit(None).alias("EmployeePositionCode"),
            lit(None).alias("EmployeePosition"),
            lit(None).alias("DepartmentCode"),
            lit(None).alias("Department"),
            lit(None).alias("TeamCode"),
            lit(None).alias("Team"),
            lit(None).alias("SupervisorPartyKey"),
            lit(None).alias("SupervisorPartyName"),
            lit(None).alias("SupervisorUserName"),
            lit(None).alias("ManagerPartyKey"),
            lit(None).alias("ManagerPartyName"),
            lit(None).alias("DirectorUsername"),
            lit(None).alias("IsEmployeeActiveFlag"),
            lit(None).alias("CreatedDate"),
            lit(None).alias("LastUpdatedDate"),
            final_solution_sixth_partition.BusinessDate.alias("BusinessDate"),
        )

        # ================== Append ==================

        firm_structure_result_df_1 = (
            firm_structure_first_partition_df.union(firm_structure_second_partition_df)
            .union(firm_structure_third_partition_df)
            .union(firm_structure_fourth_partition_df)
        )

        firm_structure_result_df_1 = firm_structure_result_df_1.distinct()

        firm_structure_result_df_2 = firm_structure_fifth_partition_df.union(
            firm_structure_sixth_partition_df
        )

        firm_structure_result_df_2 = firm_structure_result_df_2.distinct()

        firm_structure_result_df = firm_structure_result_df_2.join(
            firm_structure_result_df_1,
            (
                firm_structure_result_df_2.FirmStructureKey
                == firm_structure_result_df_1.FirmStructureKey
            ),
            "left_anti",
        )

        firm_structure_result_df = firm_structure_result_df.union(
            firm_structure_result_df_1
        )

        firm_structure_result_df.write.format("delta").clusterBy("BusinessDate").mode(
            "append"
        ).saveAsTable(curated_table_path)

    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_firm_structure_fldt(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)